# Buy & Hold vs strategie oparte na ML: BTC, SPY, TLT

Skrócona wersja analizy z pracy magisterskiej (AGH, informatyka i ekonometria). Notatnik nie uczy
modeli od nowa. Wczytuje kod pracy z `thesis_code/` i zapisane prognozy z `cache/`, a następnie
odtwarza najważniejsze wyniki. Tabele trafiają do `results/`, wykresy w `img/` pochodzą z pracy.

**Strategie porównywane na BTC (zadanie: brak dużego spadku w kolejnej sesji):**

- **Buy & Hold**: pełna ekspozycja przez cały okres.
- **Constant exposure**: stała waga BTC równa średniej ekspozycji procedury adaptacyjnej, reszta
  w gotówce. Pozwala oddzielić efekt samego zmniejszenia pozycji od efektu wyczucia momentu.
- **SMA200 filter**: prosta reguła trendu bez modelu.
- **XGBoost fixed rule**: konfiguracja stała (próg wejścia 0,55 i wyjścia 0,45, minimum 5 sesji
  w pozycji, docelowa zmienność 30%). To punkt odniesienia, a nie wariant główny.
- **XGBoost adaptive**: procedura główna. W każdym oknie testowym wybiera na podstawie
  wcześniejszych okien najlepsze konfiguracje z siatki 192 wariantów reguły i uśrednia ich pozycje.

Okres testowy: od 1 stycznia 2021 r., walidacja krocząca z rozszerzającym się oknem treningowym,
koszt 0,1% od zmiany pozycji.

In [1]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

from thesis_helpers import load_adaptive_procedure, load_notebook_code

THESIS_DIR = Path('thesis_code')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

# kod pracy: dane, cechy, reguły inwestycyjne, backtest i testy statystyczne
exec(load_notebook_code(THESIS_DIR / 'implementacja.ipynb'), globals())
# procedura adaptacyjna (w kodzie pracy: procedura_glowna); kolejność remisów przy wyborze
# konfiguracji jak w obliczeniach pracy, niezależnie od wersji numpy i procesora
load_adaptive_procedure(THESIS_DIR / 'wyniki_obliczen.ipynb', globals())
warnings.filterwarnings('ignore')

# kod pracy ustawia END_DATE na dzisiaj i przy starszym cache pobrałby dane od nowa;
# data końca danych w cache zapewnia te same wyniki co w pracy
END_DATE = pd.Timestamp('2026-09-25')

Procedura glowna — wariant: wylacznie XGBoost; 192 konfiguracji na model; liczba usrednianych wariantow wybierana z [1, 5, 10, 20, 50, 'wszystkie'].


In [2]:
# dane rynkowe i prognozy z cache (bez pobierania danych i bez ponownego uczenia)
przygotuj_dane(verbose=False)
cache = wczytaj_cache()
wf_results = cache['glowne']
robustness_runs = cache['odpornosc']
for task, res in cache['eth'].items():
    wf_results[('ETH-USD', task)] = res

TASK = 'no_large_drop'
METRICS = {'Zwrot': 'Total return', 'CAGR': 'CAGR', 'Max obsuniecie': 'Max drawdown',
           'Sharpe': 'Sharpe', 'Sortino': 'Sortino', 'Calmar': 'Calmar',
           'Zaangazowanie': 'Avg exposure'}


def summarize(name, backtest_result, ticker, oos):
    returns, positions, turnover = backtest_result
    m = compute_metrics(returns, positions, turnover, ticker=ticker,
                        asset_returns=oos['next_return'], label=name)
    return {'Strategy': name, **{new: m[old] for old, new in METRICS.items()}}


def adaptive_backtest(oos, ticker):
    return backtest(procedura_glowna(oos, ticker)['pos'], oos, ticker)


btc_oos = wf_results[('BTC-USD', TASK)]['oos']
print(f'Okres testowy BTC: {btc_oos.index.min().date()} – {btc_oos.index.max().date()}, '
      f'{len(btc_oos)} sesji')

Okres testowy BTC: 2021-01-01 – 2026-08-20, 2058 sesji


## 1. Wynik główny: BTC

Porównanie procedury adaptacyjnej z benchmarkami. Kluczowy jest wiersz *Constant exposure*:
ma tę samą średnią ekspozycję co procedura, więc różnica między nimi pokazuje, ile daje samo
wyczucie momentu wejścia i wyjścia.

In [3]:
btc_adaptive = adaptive_backtest(btc_oos, 'BTC-USD')
avg_exposure = float(btc_adaptive[1].mean())
btc_fixed_positions = build_positions(btc_oos, 'BTC-USD', 'proba_XGB', STRATEGIA_GLOWNA)

btc_strategies = {
    'Buy & Hold': buy_and_hold(btc_oos, 'BTC-USD'),
    f'Constant exposure {avg_exposure:.0%}': stala_ekspozycja(btc_oos, 'BTC-USD', avg_exposure),
    'SMA200 filter': sma_benchmark(btc_oos, 'BTC-USD', 200),
    'XGBoost fixed rule': backtest(btc_fixed_positions, btc_oos, 'BTC-USD'),
    'XGBoost adaptive': btc_adaptive,
}
btc_table = pd.DataFrame([summarize(name, res, 'BTC-USD', btc_oos)
                          for name, res in btc_strategies.items()]).set_index('Strategy')
btc_table.to_csv(RESULTS_DIR / 'btc_main.csv')
display(btc_table)

,Total return,CAGR,Max drawdown,Sharpe,Sortino,Calmar,Avg exposure
Strategy,,,,,,,
Buy & Hold,1.6644,0.1898,-0.7663,0.5327,0.7846,0.2477,1.0000
Constant exposure 39%,1.0491,0.1357,-0.3984,0.5327,0.7846,0.3406,0.3932
SMA200 filter,1.7329,0.1952,-0.6432,0.5581,0.8417,0.3035,0.5408
XGBoost fixed rule,1.0900,0.1397,-0.3518,0.5578,0.8446,0.3970,0.3194
XGBoost adaptive,2.2728,0.2340,-0.2998,0.7664,1.1910,0.7805,0.3932


![Kapitał na BTC](img/btc_equity.png)

![Obsunięcia i ekspozycja na BTC](img/btc_drawdown_exposure.png)

Na wykresach z pracy „Procedura glowna” oznacza procedurę adaptacyjną XGBoost.

## 2. Czy to coś więcej niż szczęście?

- **Test Pesarana-Timmermanna** sprawdza, czy model trafia kierunek częściej, niż wynikałoby
  z przypadku. W pracy jest 18 takich testów, więc próg po korekcie Bonferroniego wynosi 0,05/18.
- **Test Jobsona-Korkiego-Memmela** sprawdza, czy różnica wskaźnika Sharpe'a względem Buy & Hold
  jest istotna statystycznie.
- **Bootstrap stacjonarny** daje 95% przedział ufności dla wskaźnika Sharpe'a.

In [4]:
from sklearn.metrics import roc_auc_score

MODEL_NAMES = {'LR': 'Logistic regression', 'XGB': 'XGBoost', 'LSTM': 'LSTM'}
rows = []
for model in MODEL_NAMES:
    proba = btc_oos[f'proba_{model}'].values
    pt = pesaran_timmermann(btc_oos['y_true'].values, (proba > 0.5).astype(int))
    rows.append({'Model': MODEL_NAMES[model], 'AUC': roc_auc_score(btc_oos['y_true'], proba),
                 'PT p-value': pt['p_value']})
btc_forecast = pd.DataFrame(rows).set_index('Model')
bonferroni_threshold = 0.05 / 18
btc_forecast['Significant after Bonferroni'] = btc_forecast['PT p-value'] < bonferroni_threshold
display(btc_forecast)

rf_btc = rf_daily_for(btc_oos.index, FREQ_MAP['BTC-USD'])
jkm = sharpe_jk_memmel(btc_adaptive[0], btc_strategies['Buy & Hold'][0], rf=rf_btc)
ci_rows = []
for name in ['Buy & Hold', 'XGBoost adaptive']:
    ci = bootstrap_ci(btc_strategies[name][0], FREQ_MAP['BTC-USD'], rf=rf_btc)
    ci_rows.append({'Strategy': name, 'Sharpe': ci['Sharpe'],
                    'CI lower': ci['Dolna granica'], 'CI upper': ci['Gorna granica']})
btc_sharpe_ci = pd.DataFrame(ci_rows).set_index('Strategy')

print(f"Różnica Sharpe'a procedura vs Buy & Hold: p = {jkm['p_value']:.4f} (test JKM)")
display(btc_sharpe_ci)

btc_forecast.to_csv(RESULTS_DIR / 'btc_forecast_quality.csv')
btc_sharpe_ci.to_csv(RESULTS_DIR / 'btc_sharpe_ci.csv')

,AUC,PT p-value,Significant after Bonferroni
Model,,,
Logistic regression,0.5731,0.0185,False
XGBoost,0.5860,0.0003,True
LSTM,0.5147,0.6747,False


Różnica Sharpe'a procedura vs Buy & Hold: p = 0.4310 (test JKM)


,Sharpe,CI lower,CI upper
Strategy,,,
Buy & Hold,0.5327,-0.3175,1.3460
XGBoost adaptive,0.7664,0.0810,1.4777


![AUC według zadania](img/auc_by_task.png)

Kierunek jednodniowej zmiany ceny jest praktycznie nieprzewidywalny (AUC blisko 0,50). Brak dużego
spadku modele LR i XGBoost przewidują wyraźnie lepiej na BTC i SPY.

## 3. Fazy rynku na BTC

Fazy wyznaczono ex post według lat kalendarzowych. Służą tylko do opisu wyniku i nie są
sygnałem dla strategii. Obsunięcie liczone jest wewnątrz każdej fazy.

In [5]:
PHASE_NAMES = {'Hossa 2021': 'Bull 2021', 'Bessa 2022': 'Bear 2022', 'Odbicie 2023': 'Rebound 2023',
               'Hossa 2024': 'Bull 2024', '2025-2026': '2025–2026'}
rows = []
for phase, (start, end) in OKRESY.items():
    mask = (btc_oos.index >= pd.Timestamp(start)) & (btc_oos.index <= (pd.Timestamp(end) if end else btc_oos.index.max()))
    row = {'Phase': PHASE_NAMES[phase]}
    for name in ['XGBoost adaptive', 'Buy & Hold', f'Constant exposure {avg_exposure:.0%}']:
        r, p, t = (s[mask] for s in btc_strategies[name])
        m = compute_metrics(r, p, t, ticker='BTC-USD', label=name)
        row[f'{name}: return'] = m['Zwrot']
        if name != f'Constant exposure {avg_exposure:.0%}':
            row[f'{name}: max drawdown'] = m['Max obsuniecie']
    rows.append(row)
btc_phases = pd.DataFrame(rows).set_index('Phase')
btc_phases.to_csv(RESULTS_DIR / 'btc_market_phases.csv')
display(btc_phases)

,XGBoost adaptive: return,XGBoost adaptive: max drawdown,Buy & Hold: return,Buy & Hold: max drawdown,Constant exposure 39%: return
Phase,,,,,
Bull 2021,0.6336,-0.1721,0.6219,-0.5306,0.3064
Bear 2022,-0.1411,-0.2681,-0.6514,-0.6674,-0.2975
Rebound 2023,0.0667,-0.1647,1.6567,-0.2006,0.5489
Bull 2024,0.5487,-0.1040,1.1378,-0.2618,0.4370
2025–2026,0.4119,-0.2625,-0.1704,-0.5306,0.0032


![Fazy rynku](img/market_phases.png)

## 4. Inne rynki: SPY, TLT i ETH

Ta sama procedura na funduszu akcji (SPY), funduszu obligacji 20+ (TLT) i na drugiej kryptowalucie
(ETH, tylko do weryfikacji).

In [6]:
rows = []
for ticker in ['BTC-USD', 'SPY', 'TLT', 'ETH-USD']:
    oos = wf_results[(ticker, TASK)]['oos']
    adaptive = summarize('XGBoost adaptive', adaptive_backtest(oos, ticker), ticker, oos)
    passive = summarize('Buy & Hold', buy_and_hold(oos, ticker), ticker, oos)
    rows.append({'Market': ticker,
                 'Adaptive: return': adaptive['Total return'], 'B&H: return': passive['Total return'],
                 'Adaptive: Sharpe': adaptive['Sharpe'], 'B&H: Sharpe': passive['Sharpe'],
                 'Adaptive: max drawdown': adaptive['Max drawdown'],
                 'B&H: max drawdown': passive['Max drawdown']})
markets = pd.DataFrame(rows).set_index('Market')
markets.to_csv(RESULTS_DIR / 'all_markets.csv')
display(markets)

,Adaptive: return,B&H: return,Adaptive: Sharpe,B&H: Sharpe,Adaptive: max drawdown,B&H: max drawdown
Market,,,,,,
BTC-USD,2.2728,1.6644,0.7664,0.5327,-0.2998,-0.7663
SPY,0.3864,1.2297,0.2689,0.7357,-0.2185,-0.2450
TLT,-0.2906,-0.3819,-0.8630,-0.6850,-0.3425,-0.4370
ETH-USD,1.7296,2.4406,0.5394,0.6272,-0.5695,-0.7935


![Kapitał na SPY i TLT](img/spy_tlt_equity.png)

## 5. Odporność wyniku na BTC

Ta sama procedura przy zmienionych założeniach: inne ziarno generatora (W1, W2), krótsze okno
treningowe, ok. 2 lata (W3), i okres testowy od 2022 r. (W4). Na koniec koszt transakcyjny,
przy którym procedura przestaje wygrywać z Buy & Hold.

In [7]:
SCENARIO_NAMES = {'W1. ziarno 7': 'W1 seed 7', 'W2. ziarno 123': 'W2 seed 123',
                  'W3. okno kroczace 750': 'W3 rolling window 750', 'W4. OOS od 2022': 'W4 test from 2022'}
scenarios = {'W0 baseline': btc_oos}
scenarios.update({SCENARIO_NAMES[name]: run['oos'] for name, run in robustness_runs.items()})

rows = []
for name, oos in scenarios.items():
    adaptive = summarize('XGBoost adaptive', adaptive_backtest(oos, 'BTC-USD'), 'BTC-USD', oos)
    passive = summarize('Buy & Hold', buy_and_hold(oos, 'BTC-USD'), 'BTC-USD', oos)
    rows.append({'Scenario': name, 'Adaptive: Sharpe': adaptive['Sharpe'],
                 'B&H: Sharpe': passive['Sharpe'],
                 'Adaptive: max drawdown': adaptive['Max drawdown'],
                 'B&H: max drawdown': passive['Max drawdown']})
robustness = pd.DataFrame(rows).set_index('Scenario')
robustness.to_csv(RESULTS_DIR / 'btc_robustness.csv')
display(robustness)

seed_rows = [i for i in robustness.index if i.startswith(('W0', 'W1', 'W2'))]
print(f"Średni Sharpe z trzech ziaren (W0–W2): {robustness.loc[seed_rows, 'Adaptive: Sharpe'].mean():.4f} "
      f"vs Buy & Hold {robustness.loc[seed_rows[0], 'B&H: Sharpe']:.4f}")

,Adaptive: Sharpe,B&H: Sharpe,Adaptive: max drawdown,B&H: max drawdown
Scenario,,,,
W0 baseline,0.7664,0.5327,-0.2998,-0.7663
W1 seed 7,0.5524,0.5327,-0.4739,-0.7663
W2 seed 123,0.2299,0.5327,-0.3325,-0.7663
W3 rolling window 750,0.4829,0.5327,-0.3503,-0.7663
W4 test from 2022,0.4999,0.3079,-0.2250,-0.6674


Średni Sharpe z trzech ziaren (W0–W2): 0.5162 vs Buy & Hold 0.5327


In [8]:
adaptive_positions = btc_adaptive[1]
break_even_cost = np.nan
for cost in np.arange(0.0, 0.0081, 0.0001):
    adaptive_sharpe = sharpe_ann(backtest(adaptive_positions, btc_oos, 'BTC-USD', cost=cost)[0], 'BTC-USD')
    passive_sharpe = sharpe_ann(buy_and_hold(btc_oos, 'BTC-USD', cost=cost)[0], 'BTC-USD')
    if adaptive_sharpe < passive_sharpe:
        break_even_cost = cost
        break
annual_turnover = btc_adaptive[2].sum() / (len(btc_oos) / FREQ_MAP['BTC-USD'])
print(f'Koszt progowy: {break_even_cost:.2%} (w badaniu przyjęto 0,10%)')
print(f'Rotacja procedury: {annual_turnover:.1f} wartości kapitału rocznie')

Koszt progowy: 0.33% (w badaniu przyjęto 0,10%)
Rotacja procedury: 29.1 wartości kapitału rocznie


![Scenariusze odporności](img/robustness.png)

## 6. Wnioski

Notatnik wypisuje najważniejsze liczby. Interpretacja jest w README i w tekście pracy.

- Na BTC procedura adaptacyjna ma wyższy zwrot i wyższy wskaźnik Sharpe'a niż Buy & Hold przy
  znacznie płytszym obsunięciu. Wygrywa też ze stałą ekspozycją o tej samej średniej wadze.
- Różnica Sharpe'a względem Buy & Hold nie jest istotna statystycznie, a przewaga zależy od ziarna
  generatora. Ograniczenie obsunięcia jest bardziej powtarzalne niż przewaga Sharpe'a.
- Na SPY, TLT i ETH wynik się nie powtarza.

In [9]:
bh, ad = btc_table.loc['Buy & Hold'], btc_table.loc['XGBoost adaptive']
print(f"BTC: zwrot {ad['Total return']:.0%} vs {bh['Total return']:.0%}, "
      f"obsunięcie {ad['Max drawdown']:.1%} vs {bh['Max drawdown']:.1%}, "
      f"Sharpe {ad['Sharpe']:.2f} vs {bh['Sharpe']:.2f}")
print(f"Test PT dla XGBoost: p = {btc_forecast.loc['XGBoost', 'PT p-value']:.4f}; "
      f"różnica Sharpe'a: p = {jkm['p_value']:.2f}")
print(f'Tabele zapisane w {RESULTS_DIR}/')

BTC: zwrot 227% vs 166%, obsunięcie -30.0% vs -76.6%, Sharpe 0.77 vs 0.53
Test PT dla XGBoost: p = 0.0003; różnica Sharpe'a: p = 0.43
Tabele zapisane w results/
